# Housing Prices Prediction: Ames, Iowa

## Objective
Predict the final sale price of each residential property in the test set.

## Evaluation
Root Mean Squared Error on log-transformed prices:

$$\text{RMSLE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( \log(\hat{y}_i + 1) - \log(y_i + 1) \right)^2}$$

## Pipeline

| Stage | Method |
| :--- | :--- |
| **Data Cleaning** | Domain-aware null imputation, physical outlier removal |
| **Feature Engineering** | Ordinal encoding, composite area/quality features, interaction terms, spatial ratios, leakage-free target encoding, polynomial terms |
| **Transformation** | Box-Cox power transformation on skewed continuous features |
| **Validation** | 10-Fold Out-of-Fold cross-validation |
| **Base Models** | Scaled linear models, kernel methods, gradient boosted ensembles |
| **Meta-Ensemble** | Bayesian Ridge stacking + SLSQP constrained blend, CV-weighted combination |


---
## 1. Environment & Configuration

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from scipy.stats import skew
from scipy.special import boxcox1p as sp_boxcox1p

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet, BayesianRidge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.kernel_ridge import KernelRidge

warnings.filterwarnings('ignore')

SEED = 42
N_FOLDS = 10
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})

print('Environment ready.')

---
## 2. Data Ingestion

In [ ]:
paths = [
    ('/kaggle/input/competitions/home-data-for-ml-course/train.csv',
     '/kaggle/input/competitions/home-data-for-ml-course/test.csv'),
    ('/kaggle/input/home-data-for-ml-course/train.csv',
     '/kaggle/input/home-data-for-ml-course/test.csv'),
    ('train.csv', 'test.csv'),
]

train_df = test_df = None
for tr, te in paths:
    if os.path.exists(tr) and os.path.exists(te):
        train_df = pd.read_csv(tr)
        test_df = pd.read_csv(te)
        print(f'Loaded from: {tr}')
        break

if train_df is None:
    raise FileNotFoundError('Please place train.csv and test.csv in the working directory.')

print(f'Train: {train_df.shape}, Test: {test_df.shape}')

---
## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Target distribution
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='#2c3e50')
axes[0].set_title(f'SalePrice Distribution (skew={train_df["SalePrice"].skew():.2f})')

# Log-transformed target
log_price = np.log1p(train_df['SalePrice'])
sns.histplot(log_price, kde=True, ax=axes[1], color='#16a085')
axes[1].set_title(f'Log(1+SalePrice) (skew={log_price.skew():.2f})')

# GrLivArea vs SalePrice scatter with outlier markers
outlier_mask = (train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 200000)
axes[2].scatter(train_df.loc[~outlier_mask, 'GrLivArea'],
                train_df.loc[~outlier_mask, 'SalePrice'], alpha=0.4, s=12, color='#2c3e50')
axes[2].scatter(train_df.loc[outlier_mask, 'GrLivArea'],
                train_df.loc[outlier_mask, 'SalePrice'], color='red', s=50, marker='X', label='Outlier')
axes[2].set_title('GrLivArea vs SalePrice')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 4. Outlier Removal & Target Preparation

In [ ]:
# Remove Dean De Cock documented outliers
outliers = train_df[(train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 200000)].index
print(f'Removing {len(outliers)} documented outliers at indices: {list(outliers)}')
train_df = train_df.drop(outliers).reset_index(drop=True)

y_train = np.log1p(train_df['SalePrice'])
test_ids = test_df['Id']

# Combine train + test for aligned feature engineering
n_train = len(train_df)
all_data = pd.concat([
    train_df.drop(['Id', 'SalePrice'], axis=1),
    test_df.drop(['Id'], axis=1)
], axis=0).reset_index(drop=True)

print(f'Combined feature matrix: {all_data.shape}')

---
## 5. Feature Engineering

### 5.1 Missing Value Imputation
Missing values in this dataset carry domain-specific meaning. Features related to basements, garages, fireplaces, and pools use `NA` to indicate structural absence (e.g., no garage), not missing data.

In [ ]:
# --- Categorical: NA means 'feature does not exist' ---
none_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]
for c in none_cols:
    if c in all_data.columns:
        all_data[c] = all_data[c].fillna('None')

# --- Numerical: NA means 0 (structural absence) ---
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'
]
for c in zero_cols:
    if c in all_data.columns:
        all_data[c] = all_data[c].fillna(0)

# --- LotFrontage: impute using neighborhood median (spatial proxy) ---
if 'LotFrontage' in all_data.columns:
    all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
        lambda x: x.fillna(x.median())
    )

# --- Functional: per documentation, assume 'Typ' unless stated otherwise ---
if 'Functional' in all_data.columns:
    all_data['Functional'] = all_data['Functional'].fillna('Typ')

# --- Remaining: mode for categorical, median for numerical ---
for c in all_data.columns:
    if all_data[c].isnull().sum() > 0:
        if all_data[c].dtype == 'object':
            all_data[c] = all_data[c].fillna(all_data[c].mode()[0])
        else:
            all_data[c] = all_data[c].fillna(all_data[c].median())

print(f'Remaining nulls: {all_data.isnull().sum().sum()}')

### 5.2 Ordinal Encoding
Convert quality and condition scales from categorical labels to their natural numeric order.

In [ ]:
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC'
]
for c in ordinal_cols:
    if c in all_data.columns:
        all_data[c] = all_data[c].map(qual_map).fillna(0).astype(int)

exposure_map = {'Gd': 4, 'Av': 3, 'Mn': 2, 'No': 1, 'None': 0}
if 'BsmtExposure' in all_data.columns:
    all_data['BsmtExposure'] = all_data['BsmtExposure'].map(exposure_map).fillna(0).astype(int)

finish_map = {'GLQ': 6, 'ALQ': 5, 'BLQ': 4, 'Rec': 3, 'LwQ': 2, 'Unf': 1, 'None': 0}
for c in ['BsmtFinType1', 'BsmtFinType2']:
    if c in all_data.columns:
        all_data[c] = all_data[c].map(finish_map).fillna(0).astype(int)

garage_fin_map = {'Fin': 3, 'RFn': 2, 'Unf': 1, 'None': 0}
if 'GarageFinish' in all_data.columns:
    all_data['GarageFinish'] = all_data['GarageFinish'].map(garage_fin_map).fillna(0).astype(int)

functional_map = {'Typ': 8, 'Min1': 7, 'Min2': 6, 'Mod': 5, 'Maj1': 4, 'Maj2': 3, 'Sev': 2, 'Sal': 1}
if 'Functional' in all_data.columns:
    all_data['Functional'] = all_data['Functional'].map(functional_map).fillna(8).astype(int)

fence_map = {'GdPrv': 4, 'MnPrv': 3, 'GdWo': 2, 'MnWw': 1, 'None': 0}
if 'Fence' in all_data.columns:
    all_data['Fence'] = all_data['Fence'].map(fence_map).fillna(0).astype(int)

# MSSubClass is a category code, not a quantity
all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)

print('Ordinal encoding complete.')

### 5.3 Leakage-Free Target Encoding
High-cardinality categorical features like `Neighborhood` and `MSSubClass` encode strong location and building-type price signals. We apply smoothed out-of-fold target encoding to capture these without data leakage.

In [ ]:
def target_encode_oof(train_col, y, test_col, n_folds=10, smoothing=20, seed=42):
    """
    Smoothed out-of-fold target encoding.
    For each fold, the encoding is computed from the remaining folds only,
    preventing target leakage.

    Smoothing formula:
        encoded = (count * category_mean + smoothing * global_mean) / (count + smoothing)
    """
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    global_mean = y.mean()

    train_encoded = pd.Series(np.nan, index=train_col.index, dtype=float)

    for tr_idx, va_idx in kf.split(train_col):
        tr_y = y.iloc[tr_idx]
        tr_col = train_col.iloc[tr_idx]

        stats = tr_y.groupby(tr_col).agg(['mean', 'count'])
        smooth_mean = (
            (stats['count'] * stats['mean'] + smoothing * global_mean) /
            (stats['count'] + smoothing)
        )
        train_encoded.iloc[va_idx] = train_col.iloc[va_idx].map(smooth_mean).fillna(global_mean)

    # For test set, use full training data encoding
    full_stats = y.groupby(train_col).agg(['mean', 'count'])
    full_smooth = (
        (full_stats['count'] * full_stats['mean'] + smoothing * global_mean) /
        (full_stats['count'] + smoothing)
    )
    test_encoded = test_col.map(full_smooth).fillna(global_mean)

    return pd.concat([train_encoded, test_encoded]).reset_index(drop=True)

# Apply target encoding to high-cardinality categorical features
te_features = ['Neighborhood', 'MSSubClass']
for col in te_features:
    train_col = all_data[col].iloc[:n_train]
    test_col = all_data[col].iloc[n_train:]
    all_data[col + '_TE'] = target_encode_oof(train_col, y_train, test_col, smoothing=20, seed=SEED)

print(f'Target encoding applied to: {te_features}')

### 5.4 Composite & Interaction Features
Engineered features that reflect real-world drivers of property valuation.

In [ ]:
# --- Area Composites ---
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalFloorSF'] = all_data['1stFlrSF'] + all_data['2ndFlrSF']
all_data['TotalPorchSF'] = (
    all_data['WoodDeckSF'] + all_data['OpenPorchSF'] +
    all_data['EnclosedPorch'] + all_data['3SsnPorch'] + all_data['ScreenPorch']
)
all_data['TotalBathrooms'] = (
    all_data['FullBath'] + 0.5 * all_data['HalfBath'] +
    all_data['BsmtFullBath'] + 0.5 * all_data['BsmtHalfBath']
)

# --- Age & Renovation ---
all_data['HouseAge'] = (all_data['YrSold'] - all_data['YearBuilt']).clip(lower=0)
all_data['RemodAge'] = (all_data['YrSold'] - all_data['YearRemodAdd']).clip(lower=0)
all_data['GarageAge'] = (all_data['YrSold'] - all_data['GarageYrBlt']).clip(lower=0)
all_data.loc[all_data['GarageYrBlt'] == 0, 'GarageAge'] = 0  # No garage
all_data['IsRemodeled'] = (all_data['YearRemodAdd'] != all_data['YearBuilt']).astype(int)
all_data['IsNewHouse'] = (all_data['YearBuilt'] == all_data['YrSold']).astype(int)

# --- Quality x Area Interactions (strongest predictive signals) ---
all_data['Qual_TotalSF'] = all_data['OverallQual'] * all_data['TotalSF']
all_data['Qual_GrLivArea'] = all_data['OverallQual'] * all_data['GrLivArea']
all_data['ExterQual_SF'] = all_data['ExterQual'] * all_data['TotalSF']
all_data['KitchenQual_SF'] = all_data['KitchenQual'] * all_data['TotalSF']
all_data['BsmtQual_SF'] = all_data['BsmtQual'] * all_data['TotalBsmtSF']
all_data['GarageScore'] = all_data['GarageQual'] * all_data['GarageArea']

# --- Aggregate Quality Index ---
all_data['QualIndex'] = (
    all_data['OverallQual'] + all_data['ExterQual'] + all_data['KitchenQual'] +
    all_data['BsmtQual'] + all_data['GarageQual'] + all_data['FireplaceQu']
)

# --- Polynomial on OverallQual (strongest single predictor, convex price response) ---
all_data['OverallQual_Sq'] = all_data['OverallQual'] ** 2
all_data['OverallQual_Cu'] = all_data['OverallQual'] ** 3

# --- Polynomial on TotalSF and GrLivArea ---
all_data['TotalSF_Sq'] = all_data['TotalSF'] ** 2
all_data['GrLivArea_Sq'] = all_data['GrLivArea'] ** 2

# --- Location x Quality Interaction ---
all_data['Neighborhood_Qual'] = all_data['Neighborhood_TE'] * all_data['OverallQual']

# --- Spatial & Functional Ratios ---
all_data['SF_Per_Room'] = all_data['TotalSF'] / (all_data['TotRmsAbvGrd'] + 1)
all_data['LivArea_Ratio'] = all_data['GrLivArea'] / (all_data['TotalSF'] + 1)
all_data['BsmtFinRatio'] = (
    (all_data['BsmtFinSF1'] + all_data['BsmtFinSF2']) / (all_data['TotalBsmtSF'] + 1)
)
all_data['GarageToLot'] = all_data['GarageArea'] / (all_data['LotArea'] + 1)

# --- Binary Presence Indicators ---
all_data['Has2ndFloor'] = (all_data['2ndFlrSF'] > 0).astype(int)
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(int)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(int)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(int)
all_data['HasPool'] = (all_data['PoolArea'] > 0).astype(int)
all_data['HasPorch'] = (all_data['TotalPorchSF'] > 0).astype(int)

print(f'Total engineered features: {all_data.shape[1]}')

### 5.5 Near-Zero Variance Feature Removal

In [ ]:
# Drop features that are near-constant (>99.5% single value) before encoding
drop_cols = []
for c in all_data.columns:
    if all_data[c].dtype == 'object':
        top_freq = all_data[c].value_counts(normalize=True).iloc[0]
    else:
        top_freq = all_data[c].value_counts(normalize=True).iloc[0] if all_data[c].nunique() > 1 else 1.0
    if top_freq > 0.995:
        drop_cols.append(c)

print(f'Dropping {len(drop_cols)} near-zero variance features: {drop_cols}')
all_data = all_data.drop(columns=drop_cols, errors='ignore')

### 5.6 Box-Cox Skewness Correction & One-Hot Encoding

In [ ]:
num_cols = all_data.select_dtypes(include=[np.number]).columns
skewed = all_data[num_cols].apply(lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed[abs(skewed) > 0.5]

lam = 0.15
binary_feats = [
    'IsRemodeled', 'IsNewHouse', 'Has2ndFloor', 'HasGarage',
    'HasBsmt', 'HasFireplace', 'HasPool', 'HasPorch'
]
count_bc = 0
for feat in high_skew.index:
    if feat not in binary_feats:
        all_data[feat] = sp_boxcox1p(all_data[feat], lam)
        count_bc += 1

print(f'Box-Cox applied to {count_bc} features.')

# One-hot encode remaining categoricals
all_data = pd.get_dummies(all_data, drop_first=True)

X_train = all_data.iloc[:n_train].copy()
X_test = all_data.iloc[n_train:].copy()

print(f'Final feature matrix: Train {X_train.shape}, Test {X_test.shape}')

---
## 6. Validation Framework
We use `sklearn.base.clone()` to ensure each fold trains a fresh, independent model instance.

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def evaluate(model, X, y, name='Model'):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X, y)):
        m = clone(model)
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])

        pred = m.predict(X.iloc[va_idx])
        oof[va_idx] = pred
        scores.append(np.sqrt(mean_squared_error(y.iloc[va_idx], pred)))
        test_pred += m.predict(X_test) / N_FOLDS

    rmse = np.sqrt(mean_squared_error(y, oof))
    print(f'[{name:30s}] CV RMSE: {rmse:.5f}  (std: {np.std(scores):.5f})')
    return oof, test_pred, rmse

---
## 7. Base Model Training

Model selection focused on diversity: linear models capture global trends, kernel methods model non-linear surfaces, and gradient boosted trees handle complex feature interactions. Models that contributed zero weight in prior runs (Random Forest, Extra Trees) have been replaced with better-tuned alternatives.

In [ ]:
base_models = {
    'Ridge':       make_pipeline(RobustScaler(), Ridge(alpha=12.0, random_state=SEED)),
    'Lasso':       make_pipeline(RobustScaler(), Lasso(alpha=0.0003, max_iter=20000, random_state=SEED)),
    'ElasticNet':  make_pipeline(RobustScaler(), ElasticNet(alpha=0.0003, l1_ratio=0.6, max_iter=20000, random_state=SEED)),
    'BayesianRidge': make_pipeline(RobustScaler(), BayesianRidge(max_iter=500)),
    'KernelRidge': make_pipeline(RobustScaler(), KernelRidge(alpha=0.4, kernel='polynomial', degree=2, coef0=2.5)),
    'SVR':         make_pipeline(RobustScaler(), SVR(C=30, epsilon=0.006, gamma=0.0003)),
    'GBM':         GradientBoostingRegressor(
                       n_estimators=1500, learning_rate=0.01, max_depth=4,
                       max_features='sqrt', loss='huber',
                       subsample=0.75, random_state=SEED),
    'HGBM':        HistGradientBoostingRegressor(
                       max_iter=1500, learning_rate=0.01, max_depth=4,
                       l2_regularization=1.0, min_samples_leaf=10,
                       random_state=SEED),
}

try:
    from xgboost import XGBRegressor
    base_models['XGBoost'] = XGBRegressor(
        n_estimators=1500, learning_rate=0.01, max_depth=4,
        subsample=0.7, colsample_bytree=0.65,
        reg_alpha=0.3, reg_lambda=1.5,
        min_child_weight=3,
        random_state=SEED, n_jobs=-1
    )
except ImportError:
    pass

try:
    from lightgbm import LGBMRegressor
    base_models['LightGBM'] = LGBMRegressor(
        n_estimators=1500, learning_rate=0.01, max_depth=4,
        num_leaves=15, subsample=0.7, colsample_bytree=0.65,
        reg_alpha=0.3, reg_lambda=1.5,
        min_child_samples=10,
        random_state=SEED, verbose=-1
    )
except ImportError:
    pass

try:
    from catboost import CatBoostRegressor
    base_models['CatBoost'] = CatBoostRegressor(
        iterations=1500, learning_rate=0.01, depth=5,
        l2_leaf_reg=5.0, subsample=0.7,
        random_state=SEED, verbose=0
    )
except ImportError:
    pass

oof_dict = {}
test_dict = {}
score_dict = {}

print('--- Base Model Evaluation ---')
for name, model in base_models.items():
    oof_p, test_p, s = evaluate(model, X_train, y_train, name)
    oof_dict[name] = oof_p
    test_dict[name] = test_p
    score_dict[name] = s

---
## 8. Meta-Ensemble

Three independent Level-2 strategies are computed and combined using weights determined by their respective CV scores.

In [ ]:
names = list(oof_dict.keys())
OOF = np.column_stack([oof_dict[n] for n in names])
TEST = np.column_stack([test_dict[n] for n in names])

# --- Strategy A: Bayesian Ridge Meta-Stacker ---
meta = BayesianRidge(max_iter=500)
meta_oof = np.zeros(len(X_train))
meta_test = np.zeros(len(X_test))

for tr_idx, va_idx in kf.split(OOF, y_train):
    m = clone(meta)
    m.fit(OOF[tr_idx], y_train.iloc[tr_idx])
    meta_oof[va_idx] = m.predict(OOF[va_idx])
    meta_test += m.predict(TEST) / N_FOLDS

stack_rmse = np.sqrt(mean_squared_error(y_train, meta_oof))
print(f'[Bayesian Ridge Stacker       ] CV RMSE: {stack_rmse:.5f}')

# --- Strategy B: Ridge Meta-Stacker (more regularized) ---
meta_ridge = make_pipeline(RobustScaler(), Ridge(alpha=6.0, random_state=SEED))
ridge_meta_oof = np.zeros(len(X_train))
ridge_meta_test = np.zeros(len(X_test))

for tr_idx, va_idx in kf.split(OOF, y_train):
    m = clone(meta_ridge)
    m.fit(OOF[tr_idx], y_train.iloc[tr_idx])
    ridge_meta_oof[va_idx] = m.predict(OOF[va_idx])
    ridge_meta_test += m.predict(TEST) / N_FOLDS

ridge_stack_rmse = np.sqrt(mean_squared_error(y_train, ridge_meta_oof))
print(f'[Ridge Meta-Stacker           ] CV RMSE: {ridge_stack_rmse:.5f}')

# --- Strategy C: SLSQP Constrained Blend ---
def blend_loss(w):
    w = np.array(w) / np.sum(w)
    return np.sqrt(mean_squared_error(y_train, OOF @ w))

res = minimize(
    blend_loss,
    x0=[1.0 / len(names)] * len(names),
    method='SLSQP',
    bounds=[(0, 1)] * len(names),
    constraints={'type': 'eq', 'fun': lambda w: 1 - sum(w)}
)
w_opt = res.x / res.x.sum()

slsqp_oof = OOF @ w_opt
slsqp_test = TEST @ w_opt
slsqp_rmse = np.sqrt(mean_squared_error(y_train, slsqp_oof))

print(f'[SLSQP Constrained Blend      ] CV RMSE: {slsqp_rmse:.5f}')
print('\nOptimized Blend Weights:')
for n, w in zip(names, w_opt):
    if w > 0.001:
        print(f'  {n:30s}: {w*100:6.2f}%')

# --- Combine all three strategies by inverse CV RMSE ---
strategies = {
    'BayesRidge': (meta_oof, meta_test, stack_rmse),
    'RidgeStack': (ridge_meta_oof, ridge_meta_test, ridge_stack_rmse),
    'SLSQP':     (slsqp_oof, slsqp_test, slsqp_rmse),
}

inv_sum = sum(1.0 / s[2] for s in strategies.values())
final_oof = sum((1.0 / s[2]) / inv_sum * s[0] for s in strategies.values())
final_test = sum((1.0 / s[2]) / inv_sum * s[1] for s in strategies.values())
final_rmse = np.sqrt(mean_squared_error(y_train, final_oof))

print('\nMeta-Ensemble Strategy Weights:')
for k, v in strategies.items():
    w = (1.0 / v[2]) / inv_sum
    print(f'  {k:20s}: {w*100:.1f}%  (CV: {v[2]:.5f})')

print(f'\n[Final Ensemble               ] CV RMSE: {final_rmse:.5f}')

---
## 9. Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model comparison bar chart
all_scores = {**score_dict, 'BayesRidge Stack': stack_rmse, 'Ridge Stack': ridge_stack_rmse, 'SLSQP Blend': slsqp_rmse, 'Final Ensemble': final_rmse}
df_scores = pd.DataFrame({'Model': list(all_scores.keys()), 'CV RMSE': list(all_scores.values())})
df_scores = df_scores.sort_values('CV RMSE')

colors = ['darkred' if m == 'Final Ensemble' else 'steelblue' for m in df_scores['Model']]
axes[0].barh(df_scores['Model'], df_scores['CV RMSE'], color=colors)
axes[0].set_xlabel('Log-RMSE')
axes[0].set_title('Model Comparison (10-Fold CV)')
for i, (v, m) in enumerate(zip(df_scores['CV RMSE'], df_scores['Model'])):
    axes[0].text(v + 0.0005, i, f'{v:.5f}', va='center', fontsize=9)
axes[0].set_xlim(df_scores['CV RMSE'].min() - 0.01, df_scores['CV RMSE'].max() + 0.01)

# Residual analysis
residuals = y_train - final_oof
axes[1].scatter(final_oof, residuals, alpha=0.3, s=10, color='#2c3e50')
axes[1].axhline(0, color='red', linewidth=0.8)
axes[1].set_xlabel('Predicted Log(1+SalePrice)')
axes[1].set_ylabel('Residual')
axes[1].set_title(f'Residual Analysis (mean={residuals.mean():.4f})')

plt.tight_layout()
plt.show()

---
## 10. Submission

In [ ]:
predictions = np.expm1(final_test)

# Clip extreme predictions to a reasonable range
low = np.expm1(y_train).quantile(0.002)
high = np.expm1(y_train).quantile(0.998)
predictions = np.clip(predictions, low, high)

submission = pd.DataFrame({'Id': test_ids, 'SalePrice': predictions})

assert submission['SalePrice'].isnull().sum() == 0
assert (submission['SalePrice'] <= 0).sum() == 0
assert len(submission) == len(test_df)

submission.to_csv('submission.csv', index=False)

print(f'Submission saved: submission.csv ({len(submission)} rows)')
print(f'\n--- Prediction Statistics ---')
print(submission['SalePrice'].describe())
print(f'\nFirst 10 rows:')
print(submission.head(10))